# <p align = "center"> Clustering Analysis of RMSD </p>

<details open>
<summary> <strong> <ins> Key Points </ins> </strong> </summary>

- Labels-aware

    - Ligand Labels

        - Visual Identification

- Labels-agnostic

</details>


In [1]:
from pathlib import Path
rootdir = Path("../../../..").resolve()
import sys
sys.path.insert(0, str(rootdir) )
from typing import Callable, Generator

import gemmi 
import parasail
import numpy as np
from rdkit import Chem
import matplotlib.pyplot as plt

from xaidar.data.molecModels import loadPDB
from xaidar.data.molecModels import get_pdb_stats, sele_pdb, sele_Lig, get_res_CoM
from xaidar.data.molecModels import flatten_pdb, sele_AA, createPDB, get_atom_coord

/home/eoo22534/mydir/xaidar/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Ligands Visual Clustering


pymol scripts:

- Get centre of mass for one ligand

- Select ligands within x radius

    `select ligs, bymolecule (resn LIG within 2  of sele)`

- Get name of ligands

    print( cmd.get_names("objects", selection = "sele") )

In [2]:
cluster_lists= [
 ['A0608a_ligand', 'A0309a_ligand', 'A0486a_ligand', 'A0239a_ligand'],
 ['A3181b_ligand', 'A1145a_ligand', 'A0528b_ligand', 'A0900a_ligand', 'A0863a_ligand', 'A1146a_ligand', 'A1128a_ligand', 'A0514b_ligand', 'A0333a_ligand', 'A0351b_ligand', 'A0719b_ligand', 'A4596b_ligand'],
 ['A0428a_ligand', 'A0586a_ligand', 'A0305a_ligand', 'A0691a_ligand', 'A0332a_ligand', 'A2448a_ligand', 'A0541a_ligand', 'A2458a_ligand', 'A0278b_ligand', 'A0229b_ligand', 'A0354a_ligand', 'A0717a_ligand', 'A0188a_ligand'],
 ['A0229a_ligand', 'A0469a_ligand', 'A0152a_ligand', 'A0359a_ligand', 'A0228a_ligand', 'A0278a_ligand', 'A0540a_ligand', 'A0525a_ligand'],

]
print(len(cluster_lists[-1]))

8


In [3]:
lst = """A0541a
A0586a
A0278b
A0305a
A0691a
A0188a
A0229b
A0332a
A0354a
A0428a
A0717a
A2448a
A2458a""".split("\n")
print(len(lst))
print(set(lst) == set( [elem[:-7] for elem in cluster_lists[2]] ) )


13
True


In [4]:
lst = """A0359a
A0469a
A0525a
A0540a
A0278a
A0152a
A0228a
A0229a
A0922a
""".split("\n")
lst = [f"{elem}_ligand" for elem in lst if elem != ""]
print(lst)
print( str.join(" or  ", lst) )

['A0359a_ligand', 'A0469a_ligand', 'A0525a_ligand', 'A0540a_ligand', 'A0278a_ligand', 'A0152a_ligand', 'A0228a_ligand', 'A0229a_ligand', 'A0922a_ligand']
A0359a_ligand or  A0469a_ligand or  A0525a_ligand or  A0540a_ligand or  A0278a_ligand or  A0152a_ligand or  A0228a_ligand or  A0229a_ligand or  A0922a_ligand


In [5]:
lst = """A0359a
A0469a
A0525a
A0540a
A0278a
A0152a
A0228a
A0229a
A0922a
""".split("\n")
# print(lst[:-1])
lst = [f"{elem}" for elem in lst if elem != ""]
# print(lst)
print( str.join(" or  ", lst[2:-1]) )

A0525a or  A0540a or  A0278a or  A0152a or  A0228a or  A0229a


In [6]:

lst = """

""".split("\n") 
print( lst[1:-1])
# print( str.join(" or  ", lst) )

['']


In [7]:
lst = """A0900a
A1128a
""".split("\n") 
print( lst[:-1])
# print( str.join(" or  ", lst) )

['A0900a', 'A1128a']


In [8]:
# Canonical Site : 

lst = """

""".split("\n") 
print( lst[1:-1])


['']


In [9]:
# Canonical Site 1: 1 - A71EV2A-x0911/A/147/1

lst = ['A0739a', 'A0450a', 'A0451a', 'A0473a', 'A0487a', 'A0501a', 'A0501b', 'A0514a', 'A0515a', 'A0526a', 'A0528a', 'A0554a', 'A0556a', 'A0566a', 'A0310a', 'A0207a', 'A0237a', 'A0351a', 'A0365a', 'A0375a', 'A0375b', 'A0387a', 'A0416a', 'A0437a', 'A0443a', 'A0446a', 'A0719a', 'A0732a', 'A0812a', 'A0836a', 'A0836b', 'A0853a', 'A0875a', 'A0884a', 'A0911a', 'A0926a', 'A1019a', 'A1068a', 'A1080a', 'A1140a', 'A1148a', 'A1169a', 'A1180a', 'A1180b', 'A1209a', 'A1255a', 'A1292a', 'A1346a', 'A1445a', 'A1445b', 'A1775a', 'A1776a', 'A1778a', 'A1779a', 'A2290a', 'A2293a', 'A2304a', 'A2339a', 'A2351a', 'A2454a', 'A2513a', 'A2629a', 'A2846a', 'A2972a', 'A3054a', 'A3066a', 'A3175a', 'A3176a', 'A3177a', 'A3178a', 'A3181a', 'A3186a', 'A3188a', 'A3189a', 'A3191a', 'A3193a', 'A3194a', 'A3201a', 'A3207a', 'A3208a', 'A3210a', 'A3218a', 'A3221a', 'A3222a', 'A3223a', 'A3225a', 'A3229a', 'A3234a', 'A3264a', 'A3279a', 'A3280a', 'A3286a', 'A3305a', 'A3306a', 'A3324a', 'A3579a', 'A3581a', 'A3585a', 'A3587a', 'A3371a', 'A3379a', 'A3392a', 'A3403a', 'A3408a', 'A3410a', 'A3417a', 'A3419a', 'A3435a', 'A3436a', 'A3450a', 'A3480a', 'A3483a', 'A3485a', 'A3488a', 'A3489a', 'A3509a', 'A3509b', 'A3510a', 'A3593a', 'A3606a', 'A3612a', 'A3616a', 'A3616b', 'A3618a', 'A3621a', 'A3623a', 'A3634a', 'A3663a', 'A3665a', 'A3685a', 'A3694a', 'A3694b', 'A3706a', 'A3706b', 'A3521a', 'A3521b', 'A3522a', 'A3546a', 'A3548a', 'A3550a', 'A3556a', 'A3557a', 'A3557b', 'A3564a', 'A3568a', 'A3571a', 'A3571b', 'A3575a', 'A3849a', 'A3858a', 'A3858b', 'A3867a', 'A3868a', 'A3869a', 'A3883a', 'A3884a', 'A3884b', 'A3890a', 'A3895a', 'A3897a', 'A3901a', 'A3908a', 'A3914a', 'A3920a', 'A3936a', 'A3937a', 'A3938a', 'A3939a', 'A3940a', 'A3942a', 'A3944a', 'A3954a', 'A3956a', 'A3963a', 'A3963b', 'A3968a', 'A3975a', 'A3977a', 'A3982a', 'A3986a', 'A3986b', 'A3989a', 'A3990a', 'A3992a', 'A3992b', 'A3993a', 'A3993b', 'A4012a', 'A4012b', 'A4017a', 'A4017b', 'A4018a', 'A4018b', 'A4020a', 'A4020b', 'A4028a', 'A4029a', 'A4031a', 'A4031b', 'A4032a', 'A4032b', 'A4035a', 'A4035b', 'A4202a', 'A4202b', 'A4290a', 'A4291a', 'A4292a', 'A4295a', 'A4298a', 'A4303a', 'A4303b', 'A4304a', 'A4305a', 'A4306a', 'A4308a', 'A4309a', 'A4309b', 'A4310a', 'A4310b', 'A4311a', 'A4312a', 'A4313a', 'A4315a', 'A4316a', 'A4317a', 'A4319a', 'A4320a', 'A4321a', 'A4321b', 'A4322a', 'A4324a', 'A4331a', 'A4331b', 'A4333a', 'A4334a', 'A4334b', 'A4336a', 'A4341a', 'A4342a', 'A4343a', 'A4343b', 'A4344a', 'A4346a', 'A4347a', 'A4350a', 'A4351a', 'A4365a', 'A4365b', 'A4368a', 'A4372a', 'A4373a', 'A4374a', 'A4374b', 'A4380a', 'A4381a', 'A4203a', 'A4207a', 'A4207b', 'A4209a', 'A4214a', 'A4214b', 'A4237a', 'A4237b', 'A4239a', 'A4254a', 'A4260a', 'A4274a', 'A4274b', 'A4283a', 'A4283b', 'A4386a', 'A4388a', 'A4390a', 'A4402a', 'A4403a', 'A4406a', 'A4409a', 'A4415a', 'A4416a', 'A4418a', 'A4420a', 'A4421a', 'A4423a', 'A4429a', 'A4431a', 'A4445a', 'A4445b', 'A4447a', 'A4447b', 'A4449a', 'A4449b', 'A4451a', 'A4451b', 'A4456a', 'A4456b', 'A4461a', 'A4463a', 'A4464a', 'A4476a', 'A4489a', 'A4489b', 'A4490a', 'A4494a', 'A4496a', 'A4509a', 'A4510a', 'A4514a', 'A4519a', 'A4519b', 'A4524a', 'A4526a', 'A4528a', 'A4537a', 'A4538a', 'A4540a', 'A4541a', 'A4541b', 'A4544a', 'A4546a', 'A4547a', 'A4554a', 'A4559a', 'A4559b', 'A4563a', 'A4570a', 'A4571a', 'A4575a', 'A4580a', 'A4580b', 'A4583a', 'A4583b', 'A4586a', 'A4596a', 'A4596c', 'A4606a', 'A4606b', 'A4608a', 'A4611a', 'A4611b', 'A4644a', 'A4644b', 'A4650a', 'A4654a', 'A4669a', 'A4669b', 'A4673a', 'A4673b', 'A4674a', 'A4674b', 'A4684a', 'A4689a', 'A4694a', 'A4695a', 'A4701a', 'A4703a', 'A4706a', 'A4707a', 'A4708a', 'A4710a', 'A4711a', 'A4712a', 'A4715a', 'A4721a', 'A4724a', 'A4727a', 'A4735a', 'A4736a', 'A4739a', 'A4742a', 'A4742b', 'A4745a', 'A4746a', 'A4748a', 'A4751a', 'A4752a', 'A4753a', 'A4771a', 'A4777a', 'A4778a', 'A4786a', 'A4791a', 'A4794a', 'A4798a', 'A4806a', 'A4806b', 'A4809a', 'A4812a', 'A4824a', 'A4824b', 'A4826a', 'A4826b', 'A4827a', 'A4827b', 'A4829a', 'A4829b', 'A4836a', 'A4838a', 'A4839a', 'A4840a', 'A4846a', 'A4847a', 'A4855a', 'A4856a', 'A4873a', 'A4873b', 'A4874a', 'A4874b', 'A4877a', 'A4877b', 'A4880a', 'A4880b', 'A4882a', 'A4882b', 'A4883a', 'A4883b', 'A4885a', 'A4885b', 'A4894a', 'A4895a', 'A4895b', 'A4899a', 'A4899b', 'A4903a', 'A4903b', 'A4908a', 'A4918a', 'A4921a', 'A4922a', 'A4928a', 'A4934a', 'A4963a', 'A4963b', 'A4967a', 'A4967b', 'A4967c', 'A4974a', 'A4977a', 'A4977b', 'A4979a', 'A4979b', 'A4980a', 'A4986a', 'A4986b', 'A4940a', 'A4943a', 'A4943b', 'A4947a', 'A4949a', 'A4954a', 'A4959a', 'A4959b', 'A5016a', 'A5022a', 'A5022b', 'A5073a', 'A4845a', 'A4914a', 'A4992a', 'A5036a', 'A5036b', 'A5039a', 'A5039b', 'A5052a', 'A5079a', 'A5079b', 'A5081a', 'A5081b', 'A5082a', 'A5082b', 'A5090a', 'A5106a', 'A5151a', 'A5171a', 'A5171b', 'A5203a', 'A5203b', 'A5210a', 'A5210b', 'A5247a', 'A5247b', 'A5289a', 'A5289b', 'A5339a', 'A5339b', 'A5366a', 'A5366b', 'A4426a', 'A4467a', 'A4472a', 'A4516a', 'A4670a', 'A4675a', 'A4774a', 'A4784a', 'A4843a', 'A4872a', 'A4926a', 'A4935a', 'A4994a', 'A4998a', 'A5015a', 'A5015b', 'A5065a', 'A5091a', 'A5104a', 'A5150a', 'A5177a', 'A5182a', 'A5200a', 'A5212a', 'A5213a', 'A5213b', 'A5219a', 'A5219b', 'A5221a', 'A5222a', 'A5231a', 'A5249a', 'A5255a', 'A5258a', 'A5262a', 'A5262b', 'A5268a', 'A5277a', 'A5283a', 'A5283b', 'A5288a', 'A5288b', 'A5317a', 'A5359a', 'A5359b', 'A5131a', 'A5509a', 'A5525a', 'A5525b', 'A5544a', 'A5666a', 'A5383a', 'A5388a', 'A5389a', 'A5393a', 'A5394a', 'A5394b', 'A5402a', 'A5424a', 'A5424b', 'A5425a', 'A5425b', 'A5427a', 'A5449a', 'A5449b', 'A5460a', 'A5466a', 'A5475a', 'A5475b', 'A5488a', 'A5512a', 'A5512b', 'A5539a', 'A5539b', 'A5584a', 'A5584b', 'A5594a', 'A5594b', 'A5598a', 'A5598b', 'A5599a', 'A5599b', 'A5604a', 'A5608a', 'A5608b', 'A5330a', 'A5330b', 'A5351a', 'A5370a', 'A5370b', 'A5381a', 'A5381b', 'A5687a', 'A5721a', 'A5721b', 'A5836a', 'A5879a', 'A5879b', 'A5884a', 'A6738a', 'A6832a', 'A6838a', 'A7011a', 'A7014a', 'A7038a', 'A7105a', 'A7132a', 'A7147a', 'A7175a', 'A7259a', 'A7333a', 'A7459a', 'A7461a', 'A7465a', 'A7465b', 'A7515a']
# print( len(lst) )
print( str.join("_ligand or  ", lst) + "_ligand" )

A0739a_ligand or  A0450a_ligand or  A0451a_ligand or  A0473a_ligand or  A0487a_ligand or  A0501a_ligand or  A0501b_ligand or  A0514a_ligand or  A0515a_ligand or  A0526a_ligand or  A0528a_ligand or  A0554a_ligand or  A0556a_ligand or  A0566a_ligand or  A0310a_ligand or  A0207a_ligand or  A0237a_ligand or  A0351a_ligand or  A0365a_ligand or  A0375a_ligand or  A0375b_ligand or  A0387a_ligand or  A0416a_ligand or  A0437a_ligand or  A0443a_ligand or  A0446a_ligand or  A0719a_ligand or  A0732a_ligand or  A0812a_ligand or  A0836a_ligand or  A0836b_ligand or  A0853a_ligand or  A0875a_ligand or  A0884a_ligand or  A0911a_ligand or  A0926a_ligand or  A1019a_ligand or  A1068a_ligand or  A1080a_ligand or  A1140a_ligand or  A1148a_ligand or  A1169a_ligand or  A1180a_ligand or  A1180b_ligand or  A1209a_ligand or  A1255a_ligand or  A1292a_ligand or  A1346a_ligand or  A1445a_ligand or  A1445b_ligand or  A1775a_ligand or  A1776a_ligand or  A1778a_ligand or  A1779a_ligand or  A2290a_ligand or  A2293a_lig

In [10]:
cluster_lists
str.join(" or  ", cluster_lists[2]) 

'A0428a_ligand or  A0586a_ligand or  A0305a_ligand or  A0691a_ligand or  A0332a_ligand or  A2448a_ligand or  A0541a_ligand or  A2458a_ligand or  A0278b_ligand or  A0229b_ligand or  A0354a_ligand or  A0717a_ligand or  A0188a_ligand'